# ML Modeling 

Baseline focus:
- `label_sensitive_binary`
- `label_target_main`

Split settings:
- `random` for in-sample capability
- `state` for cross-state generalization


## 1) Imports


In [31]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder 
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC

from tqdm.auto import tqdm
import traceback

import yaml

## 2) Paths and Config


In [32]:
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CFG_PATH = REPO_ROOT / 'config' / 'paths.local.yaml'
if not CFG_PATH.exists():
    CFG_PATH = REPO_ROOT / 'config' / 'paths.template.yaml'
if not CFG_PATH.exists():
    raise FileNotFoundError('Missing config/paths.local.yaml and config/paths.template.yaml')

cfg = yaml.safe_load(CFG_PATH.read_text(encoding='utf-8')) or {}
shared_cfg = cfg.get('shared', {})
repo_cfg = cfg.get('repo', {})

modeling_input = shared_cfg.get('modeling_input')
if not modeling_input:
    raise ValueError('Set shared.modeling_input in YAML')

DATA_PATH = Path(modeling_input)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUNS_BASE_DIR = REPO_ROOT / repo_cfg.get('modeling_runs_dir', 'results/runs/modeling')
RUN_DIR = RUNS_BASE_DIR / f'run_{RUN_ID}'

print('Config:', CFG_PATH)
print('Dataset:', DATA_PATH)
print('Run dir:', RUN_DIR)

Config: /share/project_repo/jngoant1/WiFiMLCapstoneS2026/config/paths.local.yaml
Dataset: /share/ml-data/integrated_world0-1_merged_final.parquet
Run dir: /share/project_repo/jngoant1/WiFiMLCapstoneS2026/results/runs/modeling/run_20260414T154652Z


## 3) Run Settings


In [33]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_ROWS = 5_000_000  # None = full dataset. Set int for smoke test.

SPLIT_MODES = ['random', 'state']

STATE_N_SPLITS = 5
STATE_MAX_TRAIN_ROWS = 5_000_000  # Caps train size per fold so large models stay tractable.

DROP_UNKNOWN_TARGET = True  # Drop 'unknown' from label_target_main — labeling artifact, not a real class.

# Max vendor categories kept as distinct OHE columns; rarer vendors are merged into 'infrequent_category'. - Top 50 vendors
# See Feature Preview (§4b) for recommended threshold if running for the first time.
OHE_MAX_CATEGORIES = 50

_BASE_EXCLUDE = {'label_target_main', 'label_sensitive_binary', 'label_sensitive_subtype', 'add_state'}

## 3b) Model Registry

In [34]:
# Add new models here; they become available to MODELS_TO_RUN without touching run logic.
MODEL_REGISTRY = {
    'logreg': lambda: LogisticRegression(
        solver='saga', max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE,
    ),
    
    'sgd': lambda: SGDClassifier(
        loss='log_loss', max_iter=1000, tol=1e-4,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=5,
        learning_rate='optimal', class_weight='balanced',
        random_state=RANDOM_STATE, shuffle=True, n_jobs=-1,
    ),
    
    'rf': lambda: RandomForestClassifier(
        n_estimators=300, max_samples=0.5, random_state=RANDOM_STATE,
        class_weight='balanced_subsample', n_jobs=-1,
    ),
    'lsvc': lambda: CalibratedClassifierCV(
        LinearSVC(max_iter=2000, class_weight='balanced'),
    ),

    'lgbm': lambda: LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        class_weight='balanced', random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1,
    ),

    'hist_gbm': lambda: HistGradientBoostingClassifier(
        max_iter=100,
        early_stopping=True,
        learning_rate=0.1,
        random_state=RANDOM_STATE
    ),

    'dtree': lambda: DecisionTreeClassifier( 
        max_depth=12, # Cap depth to force generalization 
        min_samples_leaf=500, # Don't create a split unless it covers 500+ APs 
        max_features='sqrt', # Speed up training by looking at a subset of features 
        class_weight='balanced', 
        random_state=RANDOM_STATE 
    ), 

    'xgb': lambda: XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        enable_categorical=True,
        n_jobs=-1
    )


}

## 3c) Model Selection

In [35]:
# NOTE: MIGHT WANT TO UPDATE AND SEPERATE MODELS FR BINARY AND MULTI-CLASS 

# Edit this list to choose models from MODEL_REGISTRY.
MODELS_TO_RUN = ['sgd','dtree', 'lgbm', 'logreg']
# MODELS_TO_RUN = ['dtree','sgd']
# e.g. MODELS_TO_RUN = ['logreg', 'sgd']

# Feature representation for vendor_clean.
#   'ohe'        — OneHotEncoder (top OHE_MAX_CATEGORIES vendors, rest → infrequent)
#   'target_enc' — TargetEncoder (vendor → smoothed sensitivity rate; cross-fitting prevents leakage)
#   'both'       — OHE columns + target-encoded column via ColumnTransformer
FEATURE_MODE = 'both'  # 'ohe' | 'target_enc' | 'both'

## 4) Load Dataset + Quick Checkpoints


In [36]:
# 1. Load the data using Polars
if DATA_PATH.suffix == '.parquet':
    pl_df = pl.read_parquet(str(DATA_PATH))
else:
    # Adjust separator if your file is a CSV (',') or TSV ('\t')
    pl_df = pl.read_csv(str(DATA_PATH), separator='\t')

# 2. Sample for "Smoke Testing" or limit to MAX_ROWS
if MAX_ROWS is not None:
    pl_df = pl_df.sample(n=min(MAX_ROWS, pl_df.height), seed=RANDOM_STATE)

# 3. Clean and Cast columns
pl_df = pl_df.with_columns([
    pl.col('add_state').cast(pl.String).str.to_uppercase(),
    pl.col('label_sensitive_binary').cast(pl.Int64) # Ensure numeric for the model
])

# 4. Filter out missing vital info
pl_df = pl_df.filter(
    pl.col('vendor_clean').is_not_null() & 
    pl.col('add_state').is_not_null()
)

# 5. Visual Feedback / Preview
print(f"Data successfully loaded.")
print(f"Final Shape: {pl_df.shape}")
print("-" * 30)
display(pl_df.head(5))

Data successfully loaded.
Final Shape: (4999999, 5)
------------------------------


add_state,label_sensitive_binary,label_sensitive_subtype,label_target_main,vendor_clean
str,i64,str,str,str
"""NORTH CAROLINA""",0,"""non_sensitive""","""residential_street""","""Sagemcom Broadband SAS"""
"""FLORIDA""",0,"""non_sensitive""","""unknown""","""eero inc."""
"""COLORADO""",0,"""non_sensitive""","""residential_street""","""Axon Networks Inc."""
"""ILLINOIS""",0,"""non_sensitive""","""residential_street""","""Vantiva"""
"""TEXAS""",0,"""non_sensitive""","""residential_street""","""NaN"""


## 4b) Feature Preview


In [37]:
# 1. Print Column Info
print(f'All dataset columns ({len(pl_df.columns)}): {list(pl_df.columns)}')
print(f'Always excluded: {sorted(_BASE_EXCLUDE)}\n')

# 2. Calculate Vendor Value Counts
# sort by count descending to analyze the "Top" vendors
vc = pl_df['vendor_clean'].value_counts().sort('count', descending=True)
total_rows = pl_df.height

# 3. Calculate OHE Coverage (Top 50)
ohe_width = min(len(vc), OHE_MAX_CATEGORIES)
top_n_sum = vc.head(OHE_MAX_CATEGORIES)['count'].sum()
coverage = top_n_sum / total_rows

# 4. Calculate Vendors needed for 95% Coverage
target_coverage = 0.95
vc_cumsum = vc.with_columns(
    cum_pct = pl.col('count').cum_sum() / total_rows
)
n_for_target = vc_cumsum.filter(pl.col('cum_pct') <= target_coverage).height + 1

# 5. Summary Prints
print(f'vendor_clean: {len(vc)} unique vendors found.')
print(f'  Current OHE_MAX_CATEGORIES (Top {OHE_MAX_CATEGORIES}) coverage: {coverage:.1%} of rows')
print(f'  Vendors needed for {target_coverage:.0%} coverage: {n_for_target}  (current OHE_MAX_CATEGORIES={OHE_MAX_CATEGORIES})')

print(f'\nExperimental Setup:')
print(f'  Models to run: {MODELS_TO_RUN}')
print(f'  Feature Mode:  {FEATURE_MODE}')

All dataset columns (5): ['add_state', 'label_sensitive_binary', 'label_sensitive_subtype', 'label_target_main', 'vendor_clean']
Always excluded: ['add_state', 'label_sensitive_binary', 'label_sensitive_subtype', 'label_target_main']

vendor_clean: 1332 unique vendors found.
  Current OHE_MAX_CATEGORIES (Top 50) coverage: 92.6% of rows
  Vendors needed for 95% coverage: 67  (current OHE_MAX_CATEGORIES=50)

Experimental Setup:
  Models to run: ['sgd', 'dtree', 'lgbm', 'logreg']
  Feature Mode:  both


## 5) Modeling Helpers

### Preprocessing
- **`OneHotEncoder`**: expands `vendor_clean` into one 0/1 column per vendor; rare vendors (beyond `OHE_MAX_CATEGORIES`) bundled into `infrequent_category`; `sparse_output=True` keeps the matrix sparse through to the classifier

### Split strategies
- **`random`**: stratified 80/20 — class ratios preserved; tests in-sample capability
- **`state`**: `StratifiedGroupKFold` — all records from the same state stay together; test folds contain only unseen states; 5 folds = train on 4/5 of states, test on 1/5; tests geographic generalization

### Helpers
- **`build_split_folds`**: returns `[(fold_id, train_idx, test_idx)]` for the chosen split strategy
- **`evaluate_model`**: accuracy, macro/weighted F1, confusion matrix, classification report; adds PR-AUC + ROC-AUC for binary targets
- **`run_for_target`**: runs all split modes × folds × `MODELS_TO_RUN` for a single target column; returns list of result dicts

In [38]:
def build_split_folds(y, split_mode, state_series=None):
    """Return list of (fold_id, train_idx, test_idx) using integer positions."""
    
    # Generate a simple array of row numbers [0, 1, 2, ... N]
    n_samples = len(y)
    indices = np.arange(n_samples)
    
    # Convert Polars to Numpy for Scikit-Learn compatibility
    y_arr = y.to_numpy() if hasattr(y, 'to_numpy') else y

    if split_mode == 'random':
        from sklearn.model_selection import train_test_split
        train_idx, test_idx = train_test_split(
            indices, 
            test_size=TEST_SIZE, 
            random_state=RANDOM_STATE, 
            stratify=y_arr
        )
        return [('random_fold_1', train_idx, test_idx)]

    if split_mode == 'state':
        if state_series is None:
            raise ValueError("state_series is required for 'state' split mode.")

        # Clean the groups using Polars syntax
        groups = state_series.cast(pl.String).str.to_uppercase().str.strip_chars()
        
        # Identify valid rows (no nulls/empty strings); safeguard
        valid_mask = (groups.is_not_null()) & (groups != "")
        valid_indices = indices[valid_mask.to_numpy()]
        
        y_v = y_arr[valid_mask.to_numpy()]
        g_v = groups.filter(valid_mask).to_numpy()

        n_groups = len(np.unique(g_v))
        if n_groups < 3:
            raise ValueError(f'Need >=3 states for grouped CV, got {n_groups}')

        n_splits = min(STATE_N_SPLITS, n_groups)
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        folds = []
        # StratifiedGroupKFold ensures that the same state never appears in 
        # both train and test for the same fold.
        for i, (tr, te) in enumerate(splitter.split(X=np.zeros((len(y_v), 1)), y=y_v, groups=g_v), start=1):
            # Map back to the original row positions
            folds.append((f'state_fold_{i}', valid_indices[tr], valid_indices[te]))

        print(f'  State split: {n_groups} states → {n_splits} folds')
        return folds

    raise ValueError(f'Unknown split_mode: {split_mode}')

In [39]:
def evaluate_model(pipe, X_test, y_test):
    """
    Compute classification metrics. 
    For binary targets also computes PR-AUC and ROC-AUC.
    """
    # 1. Hard Predictions
    pred = pipe.predict(X_test)
    
    # 2. Base Metrics
    # Macro_f1 as our primary 'truth' since sensitivity is likely rare
    out = {
        'accuracy':    float(accuracy_score(y_test, pred)),
        'macro_f1':    float(f1_score(y_test, pred, average='macro',    zero_division=0)),
        'weighted_f1': float(f1_score(y_test, pred, average='weighted', zero_division=0)),
        
        # Convert matrix to list so it is JSON serializable for storage
        'confusion_matrix': confusion_matrix(y_test, pred).tolist(),
        'classification_report': classification_report(y_test, pred, output_dict=True, zero_division=0),
    }
    
    # 3. Probability Metrics (ROC/PR-AUC)
    # Check if binary (2 classes) and if model supports probabilities
    unique_classes = np.unique(y_test[~pd.isna(y_test)]) 
    
    if len(unique_classes) == 2 and hasattr(pipe, 'predict_proba'):
        # proba contains the confidence for the positive class (usually 1)
        proba = pipe.predict_proba(X_test)[:, 1]
        
        # PR-AUC: Focuses on the minority class (The 'Sensitive' labels)
        out['pr_auc']  = float(average_precision_score(y_test, proba))
        # ROC-AUC: General measure of class separation
        out['roc_auc'] = float(roc_auc_score(y_test, proba))
        
    return out

In [40]:
def _build_pipeline(model_name, target_type='auto'):
    """
    Constructs the ML pipeline with error handling and diagnostic logging.
    """
    try:
        # 1. Validate model existence
        if model_name not in MODEL_REGISTRY:
            raise ValueError(f"Model '{model_name}' not found in MODEL_REGISTRY.")

        # 2. Setup Encoders
        ohe = OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            max_categories=OHE_MAX_CATEGORIES,
            sparse_output=True, # Sparse is safer for 10M rows
        )
        
        te = TargetEncoder(
            target_type=target_type, 
            random_state=RANDOM_STATE,
            shuffle=True 
        )

        # 3. Build Preprocessor based on FEATURE_MODE
        if FEATURE_MODE == 'ohe':
            preprocessor = ColumnTransformer([
                ('ohe_vendor', ohe, ['vendor_clean'])
            ])
        elif FEATURE_MODE == 'target_enc':
            preprocessor = ColumnTransformer([
                ('te_features', te, ['vendor_clean', 'add_state'])
            ])
        elif FEATURE_MODE == 'both':
            preprocessor = ColumnTransformer([
                ('ohe_vendor', ohe, ['vendor_clean']),
                ('te_vendor',  te,  ['vendor_clean']),
                ('te_geography', te, ['add_state']),
            ])
        else:
            raise ValueError(f"Invalid FEATURE_MODE: {FEATURE_MODE}")

        # 4. Assemble Final Pipeline
        pipe = Pipeline(steps=[
            ('preprocessor', preprocessor), 
            ('classifier', MODEL_REGISTRY[model_name]())
        ])
        
        return pipe

    except Exception as e:
        print(f"\n[ERROR] Failed to build pipeline for model: {model_name}")
        print(f"[REASON] {str(e)}")
        # traceback.format_exc() provides the line number where it failed
        print(f"[TRACEBACK]\n{traceback.format_exc()}")
        return None

In [ ]:
def run_for_target(target_col, pl_df):
    """
    Run experiments optimized for Polars and 10M+ rows.
    """
    # 1. Efficient Filtering in Polars 
    work_df = pl_df.filter(pl.col(target_col).is_not_null())

    if target_col == 'label_target_main' and globals().get('DROP_UNKNOWN_TARGET', False):
        before = work_df.height
        work_df = work_df.filter(pl.col(target_col) != 'unknown')
        print(f'  Dropped {before - work_df.height:,} "unknown" rows')

    if target_col == 'label_sensitive_binary' and globals().get('DROP_UNKNOWN_FOR_BINARY', False):
        before = work_df.height
        work_df = work_df.filter(pl.col('label_target_main') != 'unknown')
        print(f'  Dropped {before - work_df.height:,} rows where label_target_main="unknown"')

    # Extract series for splitting 
    y_full = work_df[target_col].to_numpy()
    state_series = work_df['add_state'] # Keep as Polars series for build_split_folds

    print(f'\nTarget={target_col}, n={work_df.height:,}, models={MODELS_TO_RUN}')

    results = []
    for split_mode in SPLIT_MODES:
        folds = build_split_folds(y_full, split_mode=split_mode, state_series=state_series)

        for fold_id, train_idx, test_idx in folds:
            
            # 2. Handle state-based downsampling
            if split_mode == 'state' and STATE_MAX_TRAIN_ROWS and len(train_idx) > STATE_MAX_TRAIN_ROWS:
                rng = np.random.default_rng(RANDOM_STATE)
                train_idx = rng.choice(train_idx, size=STATE_MAX_TRAIN_ROWS, replace=False)

            # 3. SELECTING FEATURES
            # include 'add_state' because our updated build_pipeline uses it for TargetEncoding
            feature_cols = ['vendor_clean', 'add_state']
            
            # Slice in Polars (extremely fast) then convert ONLY the slice to Pandas - for memory optimization
            X_train = work_df.select(feature_cols)[train_idx].to_pandas()
            X_test  = work_df.select(feature_cols)[test_idx].to_pandas()
            y_train = y_full[train_idx]
            y_test  = y_full[test_idx]

            for model_name in MODELS_TO_RUN:
                try:
                    # Identify target type for the encoder (binary vs multiclass)
                    target_type = 'binary' if len(np.unique(y_train)) == 2 else 'multiclass'
                    
                    pipe = _build_pipeline(model_name, target_type=target_type)
                    if pipe is None: continue

                    pipe.fit(X_train, y_train)
                    metric = evaluate_model(pipe, X_test, y_test)

                    out = {
                        'target': target_col, 'split_mode': split_mode,
                        'fold_id': fold_id,   'model': model_name,
                        'feature_mode': FEATURE_MODE,
                        'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
                        **metric,
                    }
                    results.append(out)
                    print(f'  {target_col} | {split_mode} | {fold_id} | {model_name} | {FEATURE_MODE} → macro_f1={out["macro_f1"]:.4f}')
                
                except Exception as e:
                    print(f'  !!! Error running {model_name}: {e}')
                    print(traceback.format_exc())

    return results


In [41]:
def run_for_target(target_col, pl_df):
    """
    Run experiments: Model-First loop. 
    Completes all folds/splits for Model A before moving to Model B.
    """
    # 1. Efficient Filtering in Polars 
    work_df = pl_df.filter(pl.col(target_col).is_not_null())

    if target_col == 'label_target_main' and globals().get('DROP_UNKNOWN_TARGET', False):
        before = work_df.height
        work_df = work_df.filter(pl.col(target_col) != 'unknown')
        print(f'  Dropped {before - work_df.height:,} "unknown" rows')

    if target_col == 'label_sensitive_binary' and globals().get('DROP_UNKNOWN_FOR_BINARY', False):
        before = work_df.height
        work_df = work_df.filter(pl.col('label_target_main') != 'unknown')
        print(f'  Dropped {before - work_df.height:,} rows where label_target_main="unknown"')

    # XGBOOST FIX: Map string labels to integers (0, 1, 2...) 
    # This stays here so it only happens once per target, not once per model.
    work_df = work_df.with_columns(
        pl.col(target_col).cast(pl.String).cast(pl.Categorical).to_physical().alias(target_col)
    )

    # Extract series for splitting 
    y_full = work_df[target_col].to_numpy()
    state_series = work_df['add_state'] 

    print(f'\nTarget={target_col}, n={work_df.height:,}, models={MODELS_TO_RUN}')

    results = []

    # --- UPDATED LOOP ORDER: MODEL FIRST ---
    for model_name in MODELS_TO_RUN:
        print(f"\n>>> Starting all tests for model: {model_name}")
        
        for split_mode in SPLIT_MODES:
            folds = build_split_folds(y_full, split_mode=split_mode, state_series=state_series)

            for fold_id, train_idx, test_idx in folds:
                
                # 2. Handle state-based downsampling
                if split_mode == 'state' and STATE_MAX_TRAIN_ROWS and len(train_idx) > STATE_MAX_TRAIN_ROWS:
                    rng = np.random.default_rng(RANDOM_STATE)
                    train_idx = rng.choice(train_idx, size=STATE_MAX_TRAIN_ROWS, replace=False)

                # 3. SELECTING FEATURES
                feature_cols = ['vendor_clean', 'add_state']
                
                X_train = work_df.select(feature_cols)[train_idx].to_pandas().astype(str)
                X_test  = work_df.select(feature_cols)[test_idx].to_pandas().astype(str)

                # Extract y before casting it
                y_train_raw = y_full[train_idx]
                y_test_raw  = y_full[test_idx]

                # 2. Now perform the casting to fix the "Ambiguous Types" error
                y_train = np.array(y_train_raw, dtype=np.int64)
                y_test  = np.array(y_test_raw, dtype=np.int64)
                

                try:
                    # Identify target type for the encoder (binary vs multiclass)
                    target_type = 'binary' if len(np.unique(y_train)) == 2 else 'multiclass'
                    
                    pipe = _build_pipeline(model_name, target_type=target_type)
                    if pipe is None: continue

                    pipe.fit(X_train, y_train)
                    metric = evaluate_model(pipe, X_test, y_test)

                    out = {
                        'target': target_col, 'split_mode': split_mode,
                        'fold_id': fold_id,   'model': model_name,
                        'feature_mode': FEATURE_MODE,
                        'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
                        **metric,
                    }
                    results.append(out)
                    print(f'  {model_name} | {split_mode} | {fold_id} → macro_f1={out["macro_f1"]:.4f}')
                
                except Exception as e:
                    print(f'  !!! Error running {model_name} on {split_mode}/{fold_id}: {e}')
                    import traceback
                    print(traceback.format_exc())

    return results

## 6a) Run → `label_target_main`

Run this cell independently to train and evaluate on the land-use target only.


In [ ]:
# 1. Run the experiment for Land-Use (Detailed classification)
results_land_use_list = run_for_target('label_target_main', pl_df)

# 2. Convert to DataFrame for the summary/saving section
results_land_use_df = pd.DataFrame(results_land_use_list)

# 3. Add a descriptive label for easier merging later
if not results_land_use_df.empty:
    results_land_use_df['target_type'] = 'Land-Use (Detailed)'

print(f'\nTotal Land-Use runs completed: {len(results_land_use_df)}')

# 4. Quick summary display of the best performance for this target
if not results_land_use_df.empty:
    top_run = results_land_use_df.sort_values('macro_f1', ascending=False).iloc[0]
    print(f"Best Land-Use Model: {top_run['model']} ({top_run['split_mode']}) - Macro F1: {top_run['macro_f1']:.4f}")
    
    # Preview results for the Summary cell
    display(results_land_use_df.head())

  Dropped 595,905 "unknown" rows

Target=label_target_main, n=4,404,094, models=['sgd', 'dtree', 'lgbm', 'logreg']

>>> Starting all tests for model: sgd
  sgd | random | random_fold_1 → macro_f1=0.3014
  State split: 56 states → 5 folds
  sgd | state | state_fold_1 → macro_f1=0.2135
  sgd | state | state_fold_2 → macro_f1=0.2208
  sgd | state | state_fold_3 → macro_f1=0.2187
  sgd | state | state_fold_4 → macro_f1=0.2172
  sgd | state | state_fold_5 → macro_f1=0.2146

>>> Starting all tests for model: dtree
  dtree | random | random_fold_1 → macro_f1=0.2995
  State split: 56 states → 5 folds
  dtree | state | state_fold_1 → macro_f1=0.2127
  dtree | state | state_fold_2 → macro_f1=0.2033
  dtree | state | state_fold_3 → macro_f1=0.2096
  dtree | state | state_fold_4 → macro_f1=0.2359
  dtree | state | state_fold_5 → macro_f1=0.2252

>>> Starting all tests for model: lgbm


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | random | random_fold_1 → macro_f1=0.3052
  State split: 56 states → 5 folds


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | state | state_fold_1 → macro_f1=0.1998


...

  lgbm | state | state_fold_4 → macro_f1=0.2001


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | state | state_fold_5 → macro_f1=0.2504

>>> Starting all tests for model: logreg


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


  logreg | random | random_fold_1 → macro_f1=0.2912
  State split: 56 states → 5 folds


## 6b) Run → `label_sensitive_binary`

Run this cell independently to train and evaluate on the sensitive/non-sensitive binary target only.


In [42]:
# 1. Run the experiment for Privacy Sensitivity (Binary: Sensitive vs. Non-Sensitive)
results_sensitivity_list = run_for_target('label_sensitive_binary', pl_df)

# 2. Convert to DataFrame for the saving and summary section
results_sensitivity_df = pd.DataFrame(results_sensitivity_list)

# 3. Descriptive label to distinguish these results from Land-Use
if not results_sensitivity_df.empty:
    results_sensitivity_df['target_type'] = 'Privacy Sensitivity (Binary)'

print(f'\nTotal Sensitivity runs completed: {len(results_sensitivity_df)}')

# 4. Quick summary display of the best performance for this target
if not results_sensitivity_df.empty:
    top_run_sens = results_sensitivity_df.sort_values('macro_f1', ascending=False).iloc[0]
    print(f"Best Sensitivity Model: {top_run_sens['model']} ({top_run_sens['split_mode']})")
    print(f"  → Macro F1: {top_run_sens['macro_f1']:.4f}")
    print(f"  → PR-AUC:   {top_run_sens.get('pr_auc', 0):.4f}")
    
    # Preview the results dataframe
    display(results_sensitivity_df.head())


Target=label_sensitive_binary, n=4,999,999, models=['sgd', 'dtree', 'lgbm', 'logreg']

>>> Starting all tests for model: sgd
  sgd | random | random_fold_1 → macro_f1=0.6179
  State split: 56 states → 5 folds
  sgd | state | state_fold_1 → macro_f1=0.6233
  sgd | state | state_fold_2 → macro_f1=0.6213
  sgd | state | state_fold_3 → macro_f1=0.6150
  sgd | state | state_fold_4 → macro_f1=0.6123
  sgd | state | state_fold_5 → macro_f1=0.6211

>>> Starting all tests for model: dtree
  dtree | random | random_fold_1 → macro_f1=0.6299
  State split: 56 states → 5 folds
  dtree | state | state_fold_1 → macro_f1=0.6429
  dtree | state | state_fold_2 → macro_f1=0.6170
  dtree | state | state_fold_3 → macro_f1=0.6175
  dtree | state | state_fold_4 → macro_f1=0.6203
  dtree | state | state_fold_5 → macro_f1=0.6307

>>> Starting all tests for model: lgbm


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | random | random_fold_1 → macro_f1=0.5799
  State split: 56 states → 5 folds


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | state | state_fold_1 → macro_f1=0.6177


...

/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | state | state_fold_4 → macro_f1=0.6065


/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jngoant1/mlvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  lgbm | state | state_fold_5 → macro_f1=0.4957

>>> Starting all tests for model: logreg
  logreg | random | random_fold_1 → macro_f1=0.5888
  State split: 56 states → 5 folds
  logreg | state | state_fold_1 → macro_f1=0.6209
  logreg | state | state_fold_2 → macro_f1=0.6180
  logreg | state | state_fold_3 → macro_f1=0.6125
  logreg | state | state_fold_4 → macro_f1=0.6105
  logreg | state | state_fold_5 → macro_f1=0.6194

Total Sensitivity runs completed: 24
Best Sensitivity Model: dtree (state)
  → Macro F1: 0.6429
  → PR-AUC:   0.1936


,target,split_mode,fold_id,model,feature_mode,n_train,n_test,accuracy,macro_f1,weighted_f1,confusion_matrix,classification_report,pr_auc,roc_auc,target_type
0,label_sensitive_binary,random,random_fold_1,sgd,both,3999999,1000000,0.852788,0.617948,0.884677,"[[818402, 126837], [20375, 34386]]","{'0': {'precision': 0.9757086806147522, 'recal...",0.248881,0.818966,Privacy Sensitivity (Binary)
1,label_sensitive_binary,state,state_fold_1,sgd,both,3983378,1016621,0.860555,0.623273,0.890186,"[[840840, 121258], [20505, 34018]]","{'0': {'precision': 0.9761942078957909, 'recal...",0.227315,0.814894,Privacy Sensitivity (Binary)
2,label_sensitive_binary,state,state_fold_2,sgd,both,4026135,973864,0.853295,0.621312,0.884164,"[[796611, 122150], [20721, 34382]]","{'0': {'precision': 0.9746480010570979, 'recal...",0.246661,0.819905,Privacy Sensitivity (Binary)
3,label_sensitive_binary,state,state_fold_3,sgd,both,3961790,1038209,0.853279,0.615024,0.886277,"[[851316, 132722], [19605, 34566]]","{'0': {'precision': 0.9774893474838705, 'recal...",0.229139,0.821641,Privacy Sensitivity (Binary)
4,label_sensitive_binary,state,state_fold_4,sgd,both,4013548,986451,0.844863,0.612340,0.878997,"[[798699, 132593], [20442, 34717]]","{'0': {'precision': 0.9750445893930348, 'recal...",0.215077,0.812070,Privacy Sensitivity (Binary)


## 7) Summary + Save

In [1]:
# 1. Gather all results into one list
all_results_list = [
    *globals().get('results_land_use_list', []),
    *globals().get('results_sensitivity_list', []),
]

if not all_results_list:
    print("Error: No results found. Ensure you ran the Land-Use or Sensitivity execution cells.")
else:
    # 2. Build the Summary DataFrame
    # Pulling specific metrics for a scannable CSV summary
    summary = pd.DataFrame([
        {
            'target':      r['target'],
            'split_mode':  r['split_mode'],
            'fold_id':     r.get('fold_id', 'na'),
            'model':       r['model'],
            'feature_mode': r.get('feature_mode', FEATURE_MODE), 
            'n_train':     r['n_train'],
            'n_test':      r['n_test'],
            'accuracy':    r['accuracy'],
            'macro_f1':    r['macro_f1'],
            'weighted_f1': r['weighted_f1'],
            'pr_auc':      r.get('pr_auc',  float('nan')),
            'roc_auc':     r.get('roc_auc', float('nan')),
        }
        for r in all_results_list
    ]).sort_values(['target', 'split_mode', 'model', 'fold_id'])

    # 3. Create the run directory
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # 4. Save the Summary Metrics (CSV)
    summary_path = RUN_DIR / 'summary_metrics.csv'
    summary.to_csv(summary_path, index=False)

    # 5. Save the Full Detailed Results (JSON)
    # Includes confusion matrices and full classification reports
    with open(RUN_DIR / 'detailed_results.json', 'w') as f:
        json.dump(all_results_list, f, indent=2)

    # 6. Save Run Metadata
    metadata = {
        'run_id': str(RUN_ID),
        'data_path': str(DATA_PATH),
        'models_run': [str(m) for m in MODELS_TO_RUN],
        'feature_mode': FEATURE_MODE,
        'ohe_max_categories': OHE_MAX_CATEGORIES,
        'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    (RUN_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))

    print(f'Results successfully saved to: {RUN_DIR}')
    print(f'Saved: {summary_path.name}')
    print(f'Saved: detailed_results.json')
    
    # 7. Final display
    display(summary)

Error: No results found. Ensure you ran the Land-Use or Sensitivity execution cells.


## 8) Detailed Metrics (Optional)

In [ ]:
# Create a specific subfolder for export
TABLES_DIR = RUN_DIR / "detailed_tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if not all_results_list:
    print("No results available to display.")
else:
    for r in all_results_list:
        try:
            # 1. Setup metadata and naming
            target = r.get('target', 'unknown')
            mode   = r.get('split_mode', 'unknown')
            fold   = r.get('fold_id', 'na')
            model  = r.get('model', 'unknown')
            
            # Create a clean filename prefix (e.g., land_use_state_fold_1_dtree)
            file_base = f"{target}_{mode}_{fold}_{model}".replace(" ", "_").lower()
            
            label = f"{target}  |  {mode}  |  {fold}  |  {model}"
            print(f'\n{"=" * 80}\n{label}\n{"=" * 80}')

            # 2. Process Classification Report
            report_dict = r.get('classification_report', {})
            if not report_dict:
                print(f"Warning: Classification report missing for {label}")
                continue

            report_df = pd.DataFrame(report_dict).T
            report_df = report_df.apply(pd.to_numeric, errors='coerce').round(3)
            
            # SAVE Report
            report_df.to_csv(TABLES_DIR / f"{file_base}_report.csv")
            
            # 3. Process Confusion Matrix
            classes = [c for c in report_df.index if c not in ('accuracy', 'macro avg', 'weighted avg')]
            cm_data = r.get('confusion_matrix')
            
            if cm_data is not None:
                cm = pd.DataFrame(cm_data, index=classes, columns=classes)
                # Raw string r'' fixes syntax warnings for the backslash
                cm.index.name = r'Actual \ Predicted'
                
                # SAVE Confusion Matrix
                cm.to_csv(TABLES_DIR / f"{file_base}_confusion_matrix.csv")
                
                print("Confusion Matrix:")
                display(cm)
            else:
                print("Warning: Confusion matrix missing.")

            # 4. Final Display
            print("\nDetailed Metrics (Precision, Recall, F1):")
            display(report_df)
            
            print(f"\n[Files saved to: {TABLES_DIR.name}/]")
            print("\n" + "─" * 40 + "\n")

        except Exception as e:
            print(f"Error processing {target}: {e}")